## **Data Profiling**

In [ ]:
import pandas as pd
import numpy as np

#### Load Datasets

In [ ]:
years = [2011, 2012, 2014, 2018, 2021, 2023, 2024]

datasets = {}

for year in years:
    datasets[year] = pd.read_csv(
        f"../data/sparcs_{year}_raw.csv",
        low_memory=False
    )

#### Summary

In [ ]:
summary = []

for year, df in datasets.items():
    summary.append({
        "Year": year,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / 1024**2, 2)
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

#### Schema Comparison & Validation

In [ ]:
reference_columns = datasets[2024].columns

for year, df in datasets.items():

    print(f"\n{'='*60}")
    print(f"{year}")
    print("="*60)

    if reference_columns.equals(df.columns):
        print("✅ Column names and order match 2024")
    else:
        print("❌ Schema mismatch")

In [ ]:
reference = datasets[2024].columns.tolist()

for year, df in datasets.items():

    if year == 2024:
        continue

    cols = df.columns.tolist()

    print(f"\n{'='*70}")
    print(f"{year}")
    print("="*70)

    extra = sorted(set(cols) - set(reference))
    missing = sorted(set(reference) - set(cols))

    print("Extra Columns:")
    print(extra if extra else "None")

    print("\nMissing Columns:")
    print(missing if missing else "None")

In [ ]:
# -----------------------------
# Standardize All Datasets
# -----------------------------

rename_map = {
    "Facility ID": "Permanent Facility Id",
    "Hospital Service Area": "Health Service Area",
    "Zip Code - 3 digits": "Zip Code",

    "CCS Diagnosis Code": "CCSR Diagnosis Code",
    "CCS Diagnosis Description": "CCSR Diagnosis Description",

    "CCS Procedure Code": "CCSR Procedure Code",
    "CCS Procedure Description": "CCSR Procedure Description",

    "Source of Payment 1": "Payment Typology 1",
    "Source of Payment 2": "Payment Typology 2",
    "Source of Payment 3": "Payment Typology 3"
}

for year, df in datasets.items():

    # Rename columns
    df.rename(columns=rename_map, inplace=True)

    # Drop legacy column if present
    df.drop(columns=["Abortion Edit Indicator"], errors="ignore", inplace=True)
    df.to_csv(f"../data/sparcs_{year}_raw.csv", index=False)

In [ ]:
reference_columns = datasets[2024].columns

for year, df in datasets.items():

    print(f"\n{'='*60}")
    print(year)
    print("="*60)

    if reference_columns.equals(df.columns):
        print("✅ Column names and order match 2024")
    else:
        print("❌ Schema mismatch")

### Data INFO

In [ ]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"{year}")
    print("="*80)

    df.info()

### Missing Values

In [ ]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"Missing Values - {year}")
    print("="*80)

    nulls = pd.DataFrame({
        "Null Count": df.isnull().sum(),
        "Null %": (df.isnull().mean()*100).round(2)
    })

    display(nulls[nulls["Null Count"]>0].sort_values("Null Count", ascending=False))

#### Unique Values

In [ ]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"Unique Values - {year}")
    print("="*80)

    display(pd.DataFrame({
        "Column": df.columns,
        "Unique Values": df.nunique()
    }))

#### Description

In [ ]:
for year, df in datasets.items():

    print(f"\n{'='*80}")
    print(f"Describe - {year}")
    print("="*80)

    display(df.describe(include='all').T)